In [1]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,  pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from evaluate import evaluator

task_evaluator = evaluator("text-generation")

summary_prompt = """You are an expert radiologist. You are about to see the 'findings' section of a {modality} report. Please create an appropriate summary/conclusion for these findings. 
Begin your final response with "Impression: ".
Report findings: {report}
"""

modality = "CT chest"

In [2]:
df = pd.read_csv("data/mimic_rrs/mimic-CT_chest-train.csv", index_col=0)
example = df['finding'][0]
df

,finding,impression
0,again seen is a pulmonary nodule in the anteri...,stable size and appearance of soft tissue mass...
1,a necrotic left upper lung nodule involving su...,1. decrease in size of spiculated left lung no...
2,there is no pulmonary embolism. the ascending ...,1. no evidence of pulmonary embolism. 2. dilat...
3,imaging was performed from the thoracic inlet ...,1. bilateral segmental pulmonary emboli as des...
4,careful inspection of the pulmonary arteries r...,1. no evidence of pulmonary embolus. 2. large ...
...,...,...
10224,there is no evidence of pulmonary embolism or ...,"no evidence of pulmonary embolism, aortic diss..."
10225,there is no evidence of aortic dissection or p...,1. no evidence of aortic dissection or pulmona...
10226,there is no evidence of pulmonary embolism. th...,1) no evidence of pulmonary embolism. 2) incre...
10227,"within the lung parenchyma, there is bilateral...",no acute traumatic intrathoracic or intra-abdo...


In [3]:
# model_id = "google/medgemma-1.5-4b-it"
model_id = "openai/gpt-oss-20b"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=quantization_config
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

llm = HuggingFacePipeline(pipeline=pipe)

llm_= ChatHuggingFace(llm=llm)

output = llm.invoke(f"Summarize the following report: {summary_prompt.format(modality=modality, report=example)}")


print(output)




ValueError: The model is quantized with Mxfp4Config but you are passing a BitsAndBytesConfig config. Please make sure to pass the same quantization config class to `from_pretrained` with different loading attributes.

In [5]:
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
print("pipeline module:", pipeline.__module__)

transformers version: 5.3.0
transformers file: /home/stephen/langchain/lib/python3.11/site-packages/transformers/__init__.py
pipeline module: transformers.pipelines
